# 🤖 Interactive Text-to-SQL Assistant ("DataTalker AI")

Welcome to the **Natural Language to SQL Assistant** walkthrough for the Yipit Analytical Data Platform!

This notebook demonstrates the complete **End-to-End Hybrid AI + Data Engineering Pipeline**:
1. **DuckDB Schema Provider**: Dynamically inspects the dimensional warehouse tables (`dim_company`, `fct_article`, `fct_arr_observation`, etc.).
2. **Gemini AI Query Writer**: Translates your plain English questions into valid DuckDB SQL queries.
3. **Security Bouncer (Safety Guard)**: Validates queries for Read-Only safety, blocks destructive commands (`DROP`, `DELETE`), and injects row limits.
4. **DuckDB Execution & Executive Summary**: Executes the approved SQL on your local data and explains the answer in plain English.

## ⚙️ Step 0: Imports & Path Configuration

In [ ]:
import os
import sys

# Ensure parent directory is in Python path for local package imports
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ai_text_to_sql.schema_provider import init_duckdb_warehouse, get_warehouse_schema_context
from ai_text_to_sql.sql_guard import clean_sql_query, validate_sql_safety, enforce_row_limit
from ai_text_to_sql.sql_generator import get_gemini_client, generate_sql, explain_results, ask_warehouse

print("✅ All modules successfully imported!")

✅ All modules successfully imported!


## 🗄️ Step 1: Initialize DuckDB & Inspect Warehouse Tables

Let's initialize our in-memory DuckDB database and load all 5 warehouse tables directly from `data/warehouse/*.csv`.

In [ ]:
con = init_duckdb_warehouse()

print("📊 Modeled Warehouse Tables Loaded in DuckDB:")
tables = ['dim_company', 'fct_article', 'fct_arr_observation', 'agg_company_quarterly_arr', 'view_company_latest_arr']
for t in tables:
    count = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"  • Table {t:<28}: {count:>4} rows")

📊 Modeled Warehouse Tables Loaded in DuckDB:
  • Table dim_company                 :   26 rows
  • Table fct_article                 :  750 rows
  • Table fct_arr_observation         :  558 rows
  • Table agg_company_quarterly_arr   :  315 rows
  • Table view_company_latest_arr     :   26 rows


### View the Generated Schema Prompt for Gemini
Here is the exact schema context string generated dynamically for the LLM:

In [ ]:
schema_prompt = get_warehouse_schema_context(con, sample_limit=1)
print(schema_prompt[:1200] + "\n... [TRUNCATED DISPLAY] ...")

### DATABASE ENGINE: DuckDB SQL (Case-Insensitive Table and Column Names)
### AVAILABLE WAREHOUSE TABLES & SCHEMAS:

#### Table: `dim_company`
**Purpose**: Master company dimension with industry, headquarters, headcount, founded year, public status, and size category.
**Columns**: `company_id` (VARCHAR), `company_name` (VARCHAR), `industry` (VARCHAR), `headquarters` (VARCHAR), `founded_year` (BIGINT), `employee_count` (BIGINT), `company_size_category` (VARCHAR), `is_public` (BOOLEAN), `stock_ticker` (VARCHAR), `has_company_metadata` (BOOLEAN)
**Sample Rows (1)**:
  - {'company_id': 'COMP001', 'company_name': 'Airbnb', 'industry': 'Data Analytics', 'headquarters': 'London, UK', 'founded_year': 1999, 'employee_count': 19967, 'company_size_category': 'Medium', 'is_public': False, 'stock_ticker': None, 'has_company_metadata': True}

#### Table: `fct_article`
**Purpose**: Tech news articles with publication dates, standardized categories, authors, summaries, and source URLs.
**Columns**: `a

## 🛡️ Step 2: Testing the SQL Safety Guard

Before executing any LLM-generated code against our database, `sql_guard.py` acts as a security bouncer:
* Enforces **Read-Only** (`SELECT` / `WITH` statements only)
* Blocks dangerous mutating operations (`DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`)
* Injects a safe `LIMIT 50` clause

In [ ]:
test_cases = [
    ("Safe Query", "SELECT company_name, arr_usd_M FROM view_company_latest_arr ORDER BY arr_usd_M DESC"),
    ("Malicious: DROP", "DROP TABLE dim_company;"),
    ("Malicious: Semicolon Injection", "SELECT 1; DELETE FROM fct_article;"),
]

for label, query in test_cases:
    is_safe, error = validate_sql_safety(query)
    status = "✅ APPROVED" if is_safe else "🚫 BLOCKED"
    print(f"{status} | {label:<30} -> {error or 'Safe to run'}")

✅ APPROVED | Safe Query                     -> Safe to run
🚫 BLOCKED | Malicious: DROP                -> Security Violation: Statement type 'DROP' is prohibited. Only SELECT queries are permitted.
🚫 BLOCKED | Malicious: Semicolon Injection -> Security Violation: Multiple SQL statements detected. Only a single query is allowed.


## 🚀 Step 3: Run End-to-End Business Questions

Now let's ask real business questions. Watch how the pipeline:
1. Translates the English question into DuckDB SQL
2. Validates safety & row limits
3. Runs in DuckDB and prints the dataframe
4. Returns a 2-sentence executive summary

In [ ]:
def ask(question: str):
    print(f"\n{'='*80}")
    print(f"❓ QUESTION: {question}")
    print(f"{'='*80}")
    
    res = ask_warehouse(question, con, schema_prompt)
    
    if not res["success"]:
        print(f"❌ ERROR: {res['error']}")
        return
        
    print("\n📝 GENERATED DUCKDB SQL:")
    print(res["sql"])
    
    print("\n📊 QUERY RESULTS:")
    df = res["results_df"]
    display(df) if "display" in globals() else print(df)
    
    print("\n💡 EXECUTIVE SUMMARY:")
    print(res["summary"])

# Test Question 1
ask("Which top 5 companies have the highest latest ARR?")


❓ QUESTION: Which top 5 companies have the highest latest ARR?

📝 GENERATED DUCKDB SQL:
SELECT 
    company_name,
    industry,
    latest_arr_usd_M,
    latest_observation_date
FROM view_company_latest_arr
ORDER BY latest_arr_usd_M DESC
LIMIT 5

📊 QUERY RESULTS:
          company_name         industry  latest_arr_usd_M  \
0  Amazon Web Services             SaaS            102600   
1                Tesla  Cloud Computing             98880   
2            Microsoft   Data Analytics             83850   
3               NVIDIA            AI/ML             62100   
4                 Uber          FinTech             36380   

  latest_observation_date  
0              2024-10-04  
1              2024-07-14  
2              2024-12-02  
3              2024-10-14  
4              2024-10-09  

💡 EXECUTIVE SUMMARY:
The top 5 companies by latest ARR are led by Amazon Web Services ($102.6B) and Tesla ($98.88B), followed by Microsoft ($83.85B), NVIDIA ($62.1B), and Uber ($36.38B). Amazon Web S

In [ ]:
# Test Question 2
ask("What are the top 3 tech categories by total news article count?")


❓ QUESTION: What are the top 3 tech categories by total news article count?

📝 GENERATED DUCKDB SQL:
SELECT 
    category_clean AS category,
    COUNT(*) AS article_count
FROM fct_article
WHERE category_clean IS NOT NULL
GROUP BY category_clean
ORDER BY article_count DESC
LIMIT 3

📊 QUERY RESULTS:
         category  article_count
0           AI_ML            161
1  Data_Analytics            131
2         FinTech            124

💡 EXECUTIVE SUMMARY:
The top three tech categories by total news article count are AI/ML with 161 articles, followed by Data Analytics with 131 and FinTech with 124. These three sectors dominate current media coverage within the dataset.


In [ ]:
# Test Question 3
ask("Show the quarterly ARR revenue trend for OpenAI.")

## 💬 Step 4: Try Your Own Questions!

Type any question in the cell below and run it to chat with your warehouse dataset!

In [ ]:
# Replace with any question you want to ask your data:
my_question = "Which companies are headquartered in London, UK, and what are their industries?"

ask(my_question)